# Lund RNTuple histograms

The notebook twin of [`cpp/apps/hist_lund_rntuple.cpp`](../cpp/apps/hist_lund_rntuple.cpp):
it reads a `jets.root` **RNTuple** with `uproot` and fills the same histograms the C++ app
writes, **bin for bin** — same edges, same per-jet `weight`, same `x_nsec > 0` gate on the
secondary-$k_t$ columns. Use the app when you want a ROOT file to post-process; use this
when you want to look at the distributions.

Figure order:

1. the two **primary Lund planes** side by side — hadron level $x$ and parton level $y$;
2. the **jet quantities** — $p_T$ (linear and log-binned), mass, $\eta$, $\phi$;
3. the primary **multiplicity**, the four Lund observables **pooled over splittings**, the
   same four **split by splitting index** (the first two only — set `NSPLIT_SHOW` to see
   more), and every **aux** conditioning column.

Throughout, **blue is hadron level ($x$)** and **orange is parton level ($y$)**; the two
levels share axis definitions, so any panel can be divided or subtracted without rebinning.

## Setup

`BINS` below is the C++ app's binning transcribed. Keep the two in sync if you change either
— the parity check in the last section is what catches drift.

In [ ]:
from pathlib import Path

import awkward as ak
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import uproot

# --- what to read (mirrors the app's command line) ---------------------------
INPUT = Path("../cpp/test_data/jets_aux.root")
NTUPLE = "Jets"
NSPLIT_SHOW = 2  # per-splitting figures for index 0..NSPLIT_SHOW-1   (--nsplit)
PTMAX = 1000.0  # upper edge of the jet_pt / x_ptg axes              (--ptmax)
MMAX = 200.0  # upper edge of the jet_m / x_mg axes                (--mmax)

# --- binning: identical to hist_lund_rntuple.cpp -----------------------------
BINS = {
    "lnInvDelta": (100, 0.0, 10.0),
    "lnkt": (120, -4.0, 8.0),
    "lnz": (100, -10.0, 0.0),
    "psi": (100, -np.pi, np.pi),
    "mult": (51, -0.5, 50.5),  # also x_nsec and x_sec_attach
    "jet_pt": (200, 0.0, PTMAX),
    "jet_eta": (100, -5.0, 5.0),
    "jet_phi": (100, -np.pi, np.pi),
    "jet_m": (200, 0.0, MMAX),
    "x_mg": (200, 0.0, MMAX),
    "x_ptg": (200, 0.0, PTMAX),
    "x_kt_sec_max": (200, 0.0, 100.0),
    "x_kt_sec_sum": (200, 0.0, 200.0),
}
# The log-binned pt twin: 200 bins from 10 GeV to max(PTMAX, 20), decade-uniform.
PT_LOG_EDGES = np.logspace(np.log10(10.0), np.log10(max(PTMAX, 20.0)), 201)

LABEL = {
    "lnInvDelta": r"$\ln(1/\Delta R)$",
    "lnkt": r"$\ln(k_t/\mathrm{GeV})$",
    "lnz": r"$\ln z$",
    "psi": r"$\psi$",
}

In [ ]:
# --- style -------------------------------------------------------------------
# Two series only, so two categorical slots, assigned by entity and never swapped:
# blue = hadron level (x), orange = parton level (y). The Lund-plane density is a
# magnitude, so it takes the one-hue sequential blue ramp (light -> dark).
C_X, C_Y = "#2a78d6", "#eb6834"
SURFACE, INK, INK_2, MUTED, GRID, AXIS = (
    "#fcfcfb",
    "#0b0b0b",
    "#52514e",
    "#898781",
    "#e1e0d9",
    "#c3c2b7",
)
SEQ_BLUE = [
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5",
    "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b",
]
CMAP = mpl.colors.LinearSegmentedColormap.from_list("h2p_blue", SEQ_BLUE)
CMAP.set_bad(SURFACE)  # empty bins recede to the surface instead of reading as data

mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "figure.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": AXIS,
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlesize": 9,
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelcolor": INK_2,
        "ytick.labelcolor": INK_2,
        "grid.color": GRID,
        "grid.linestyle": "-",  # solid hairline; dashing reads as "threshold"
        "grid.linewidth": 0.6,
        "font.size": 9,
        "legend.frameon": False,
        "lines.linewidth": 1.6,
    }
)


def edges(key):
    n, lo, hi = BINS[key]
    return np.linspace(lo, hi, n + 1)


def h1(values, weights, key):
    """The app's TH1F::Fill loop: weighted counts on the app's edges."""
    e = edges(key)
    return np.histogram(values, bins=e, weights=weights)[0], e


def step(ax, counts, e, color, label=None):
    ax.stairs(counts, e, color=color, linewidth=1.6, label=label)


def zoom(ax, counts, e, pad=0.03):
    """Keep the app's binning, view only the populated range.

    The stored ranges are sized for a harder sample than the one loaded here, so
    plotting them whole would squeeze every distribution into the leftmost bins.
    Nothing is rebinned or dropped — this is a view limit.
    """
    tot = np.sum(np.atleast_2d(counts), axis=0)
    hit = np.flatnonzero(tot > 0)
    if hit.size == 0:
        return
    lo, hi = e[hit[0]], e[hit[-1] + 1]
    if ax.get_xscale() == "log":  # call after set_xscale; margins go multiplicative
        f = (hi / lo) ** pad
        ax.set_xlim(lo / f, hi * f)
    else:
        m = pad * (hi - lo)
        ax.set_xlim(lo - m, hi + m)


def finish(ax, xlabel, title="", ylabel="weighted jets", logy=False):
    # Falling GeV-scale spectra get a log count axis; their tails run three decades
    # below the peak and are invisible otherwise.
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y")
    ax.set_axisbelow(True)

## Load

`uproot` reads the RNTuple directly — no ROOT install needed on the Python side. The
provenance printed below is the app's `meta/` directory: written per entry by
[`lund_writer.cpp`](../cpp/src/lund_writer.cpp) and constant by construction.

In [ ]:
f = uproot.open(INPUT)
nt = f[NTUPLE]
present = set(nt.keys())

# The aux conditioning columns post-date the original schema; a file written before
# docs/PLAN_Input.md simply lacks them and must still histogram.
HAS_AUX = {"x_mg", "x_nsec"} <= present
HAS_PTG = "x_ptg" in present
HAS_SEC = {"x_kt_sec_max", "x_kt_sec_sum", "x_sec_attach"} <= present

cols = [
    "weight", "jet_pt", "jet_eta", "jet_phi", "jet_m",
    "x_lnInvDelta", "x_lnkt", "x_lnz", "x_psi",
    "y_lnInvDelta", "y_lnkt", "y_lnz", "y_psi",
]
cols += [c for c in ("x_mg", "x_nsec", "x_ptg", "x_kt_sec_max", "x_kt_sec_sum",
                     "x_sec_attach") if c in present]
ev = nt.arrays(cols)
w = ak.to_numpy(ev["weight"])
meta = nt.arrays(["generator", "z_cut", "beta", "kt_floor"], entry_stop=1)

print(f"{INPUT}:{NTUPLE}")
print(f"  jets      : {len(w)}   (sum of weights {w.sum():.6g})")
print(f"  generator : {meta['generator'][0]}")
print(f"  grooming  : z_cut={meta['z_cut'][0]:.3f}  beta={meta['beta'][0]:.3f}  "
      f"kt_floor={meta['kt_floor'][0]:.3f} GeV")
print(f"  aux       : {'present' if HAS_AUX else 'absent (pre-PLAN_Input file)'}")


def pooled(level, obs):
    """All splittings of one level, flattened, with the jet weight broadcast onto each."""
    jag = ev[f"{level}_{obs}"]
    wj = ak.broadcast_arrays(ak.Array(w), jag)[0]
    return ak.to_numpy(ak.flatten(jag)), ak.to_numpy(ak.flatten(wj))


def nth(level, obs, n):
    """Splitting index `n` only — the app's splitNN/ directories."""
    jag = ev[f"{level}_{obs}"]
    keep = ak.to_numpy(ak.num(jag) > n)
    return ak.to_numpy(jag[keep][:, n]), w[keep]


LEVELS = [("x", "hadron level", C_X), ("y", "parton level", C_Y)]

## 1 · The primary Lund planes

The headline panel: the density of primary splittings in $(\ln 1/\Delta R,\ \ln k_t)$ at both
levels, on a **shared colour scale** so the two are read against each other directly. The
hard lower-left edges are the pipeline's own cuts — the Soft Drop boundary and the
perturbative $\ln k_t$ floor printed above — not a feature of the shower.

In [ ]:
xe, ye = edges("lnInvDelta"), edges("lnkt")
planes = {}
for lvl, _, _ in LEVELS:
    u, wu = pooled(lvl, "lnInvDelta")
    k, _ = pooled(lvl, "lnkt")
    planes[lvl] = np.histogram2d(u, k, bins=[xe, ye], weights=wu)[0] / w.sum()

vmax = max(h.max() for h in planes.values())

fig, axs = plt.subplots(1, 2, figsize=(9.2, 3.9), sharex=True, sharey=True)
for ax, (lvl, name, _) in zip(axs, LEVELS):
    h = np.ma.masked_where(planes[lvl] == 0, planes[lvl])
    im = ax.pcolormesh(xe, ye, h.T, cmap=CMAP, vmin=0, vmax=vmax, rasterized=True)
    ax.set_title(f"{name}  ({lvl})")
    ax.set_xlabel(LABEL["lnInvDelta"])
axs[0].set_ylabel(LABEL["lnkt"])

# View the populated corner; the stored plane is 0..10 x -4..8 as in the app.
occ = sum(planes.values())
xs, ys = np.flatnonzero(occ.sum(1) > 0), np.flatnonzero(occ.sum(0) > 0)
axs[0].set_xlim(xe[xs[0]], xe[xs[-1] + 1])
axs[0].set_ylim(ye[ys[0]], ye[ys[-1] + 1])

cb = fig.colorbar(im, ax=axs, fraction=0.046, pad=0.02)
cb.set_label("splittings / jet / bin", color=INK_2)
cb.outline.set_visible(False)
fig.suptitle("Primary Lund planes, hadron vs parton level", x=0.065, ha="left",
             fontsize=11)
plt.show()

## 2 · Jet quantities

The jets these sequences were declustered from: the hadron-level anti-$k_t$ jets that
[`lund_io.cpp`](../cpp/src/lund_io.cpp) matched to a parton-level partner. The log-binned
$p_T$ twin is the one to read for the jet-$p_T$ dependence of everything downstream
([`docs/PLAN_jet_xsection.md`](../docs/PLAN_jet_xsection.md)); the linear one is the same
data on the app's linear axis.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10.4, 5.4))
axs = axs.ravel()

c, e = h1(ak.to_numpy(ev["jet_pt"]), w, "jet_pt")
step(axs[0], c, e, C_X)
zoom(axs[0], c, e)
finish(axs[0], r"$p_T$ [GeV]", "jet $p_T$", logy=True)

c_log = np.histogram(ak.to_numpy(ev["jet_pt"]), bins=PT_LOG_EDGES, weights=w)[0]
step(axs[1], c_log, PT_LOG_EDGES, C_X)
axs[1].set_xscale("log")
zoom(axs[1], c_log, PT_LOG_EDGES)
finish(axs[1], r"$p_T$ [GeV]", "jet $p_T$ (log bins)", logy=True)

c, e = h1(ak.to_numpy(ev["jet_m"]), w, "jet_m")
step(axs[2], c, e, C_X)
zoom(axs[2], c, e)
finish(axs[2], r"$m$ [GeV]", "jet mass", logy=True)

c, e = h1(ak.to_numpy(ev["jet_eta"]), w, "jet_eta")
step(axs[3], c, e, C_X)
zoom(axs[3], c, e)
finish(axs[3], r"$\eta$", r"jet $\eta$")

c, e = h1(ak.to_numpy(ev["jet_phi"]), w, "jet_phi")
step(axs[4], c, e, C_X)
finish(axs[4], r"$\phi$", r"jet $\phi$")
axs[4].set_ylim(bottom=0)

axs[5].set_visible(False)
fig.suptitle("Jet kinematics (hadron level)", x=0.045, ha="left", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## 3 · Primary multiplicity

How many splittings survive grooming on the spine, per jet, at each level. The parton-level
sequence is the shorter of the two — the target the model has to produce from the longer
hadron-level input.

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.4))
for lvl, name, colour in LEVELS:
    n = ak.to_numpy(ak.num(ev[f"{lvl}_lnkt"]))
    c, e = h1(n, w, "mult")
    step(ax, c, e, colour, label=f"{name} ({lvl}), mean {np.average(n, weights=w):.2f}")
    zoom(ax, c, e)
finish(ax, r"$n_\mathrm{splittings}$", "primary multiplicity")
ax.legend()
plt.show()

## 4 · Lund observables, pooled over splittings

Every splitting of every jet, both levels overlaid — the app's `*_all_*` histograms.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(11.4, 2.9))
for ax, obs in zip(axs, LABEL):
    cs = []
    for lvl, name, colour in LEVELS:
        v, wv = pooled(lvl, obs)
        c, e = h1(v, wv, obs)
        step(ax, c, e, colour, label=f"{name} ({lvl})")
        cs.append(c)
    zoom(ax, cs, e)
    finish(ax, LABEL[obs], ylabel="weighted splittings")
axs[3].set_ylim(bottom=0)
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncols=2, bbox_to_anchor=(0.99, 1.02))
fig.suptitle("All splittings pooled", x=0.04, ha="left", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

## 5 · Split by splitting index

The same four observables for the **first two splittings** separately (`NSPLIT_SHOW = 2`;
the app books eight by default). Each row conditions on the jet having at least that many
splittings, so the row populations differ — the count in each panel title is how many jets
reach that index at that level.

In [ ]:
fig, axs = plt.subplots(NSPLIT_SHOW, 4, figsize=(11.4, 2.9 * NSPLIT_SHOW),
                        squeeze=False)
for n in range(NSPLIT_SHOW):
    reach = [int(np.count_nonzero(ak.to_numpy(ak.num(ev[f"{lvl}_lnkt"])) > n))
             for lvl, _, _ in LEVELS]
    for j, (ax, obs) in enumerate(zip(axs[n], LABEL)):
        cs = []
        for lvl, name, colour in LEVELS:
            v, wv = nth(lvl, obs, n)
            c, e = h1(v, wv, obs)
            step(ax, c, e, colour, label=f"{name} ({lvl})")
            cs.append(c)
        zoom(ax, cs, e)
        ylabel = f"splitting {n}\nweighted jets" if j == 0 else "weighted jets"
        finish(ax, LABEL[obs], ylabel=ylabel)
    # How many jets reach this index, per level — outside the axes so it cannot
    # collide with the distribution.
    axs[n][0].set_title(f"x {reach[0]} · y {reach[1]} jets", loc="right", fontsize=8,
                        color=INK_2)
    axs[n][3].set_ylim(bottom=0)
handles, labels = axs[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncols=2, bbox_to_anchor=(0.99, 1.02))
fig.suptitle("Per-splitting breakdown", x=0.04, ha="left", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

## 6 · Aux conditioning observables

The hadron-level all-branch scalars of [`docs/PLAN_Input.md`](../docs/PLAN_Input.md) — the
conditioning-side information the primary-only sequence structurally cannot carry.

`x_kt_sec_max`, `x_kt_sec_sum` and `x_sec_attach` are **undefined when `x_nsec == 0`**
(written as 0, per [`lund_io.hpp`](../cpp/include/lund_io.hpp)), so those three are filled
only from jets with at least one off-spine passing splitting — exactly as the app gates
them. Filling them unconditionally would fabricate a spike at zero.

In [ ]:
if not HAS_AUX:
    print("no aux columns in this file — nothing to plot (pre-PLAN_Input schema)")
else:
    nsec = ak.to_numpy(ev["x_nsec"])
    sec = nsec > 0
    print(f"secondary-kt panels use {sec.sum()} / {len(nsec)} jets with x_nsec > 0")

    # (binning key, values, weights, x label, title, log count axis)
    panels = [("x_mg", ak.to_numpy(ev["x_mg"]), w, r"$m_g$ [GeV]", "groomed jet mass",
               True)]
    if HAS_PTG:
        panels.append(("x_ptg", ak.to_numpy(ev["x_ptg"]), w, r"$p_{T,g}$ [GeV]",
                       "groomed jet $p_T$", True))
    panels.append(("mult", nsec, w, r"$n_\mathrm{sec}$", "secondary splittings", False))
    if HAS_SEC:
        panels += [
            ("x_kt_sec_max", ak.to_numpy(ev["x_kt_sec_max"])[sec], w[sec],
             r"$k_t^\mathrm{sec,max}$ [GeV]", r"hardest secondary $k_t$", True),
            ("x_kt_sec_sum", ak.to_numpy(ev["x_kt_sec_sum"])[sec], w[sec],
             r"$\Sigma k_t^\mathrm{sec}$ [GeV]", r"summed secondary $k_t$", True),
            ("mult", ak.to_numpy(ev["x_sec_attach"])[sec], w[sec], "node index",
             "hardest-secondary attachment", False),
        ]

    ncol = 3
    nrow = int(np.ceil(len(panels) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(10.4, 2.7 * nrow), squeeze=False)
    axs = axs.ravel()
    for ax, (key, v, wv, xlabel, title, logy) in zip(axs, panels):
        c, e = h1(v, wv, key)
        step(ax, c, e, C_X)
        zoom(ax, c, e)
        gated = r"  ($n_\mathrm{sec}>0$)" if len(wv) < len(w) else ""
        finish(ax, xlabel, title + gated, logy=logy)
    for ax in axs[len(panels):]:
        ax.set_visible(False)
    fig.suptitle("Aux conditioning columns (hadron level)", x=0.045, ha="left",
                 fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()

## 7 · Parity with the C++ app *(optional)*

If [`hist_lund_rntuple`](../cpp/apps/hist_lund_rntuple.cpp) has been run on the same input,
compare bin contents directly. Anything other than zero means the two binnings have drifted.

```sh
cd cpp/build && cmake .. && make hist_lund_rntuple
./hist_lund_rntuple ../test_data/jets_aux.root lund_hists.root
```

In [ ]:
HIST_ROOT = Path("../cpp/build/lund_hists.root")

if not HIST_ROOT.exists():
    print(f"{HIST_ROOT} not found — skipping the parity check (see the cell above)")
else:
    ref = uproot.open(HIST_ROOT)
    worst = 0.0
    rows = []

    def compare(path, mine):
        global worst
        theirs = ref[path].values()
        d = float(np.abs(theirs - mine).max())
        worst = max(worst, d)
        rows.append((path, d))

    compare("jet/h_jet_pt", h1(ak.to_numpy(ev["jet_pt"]), w, "jet_pt")[0])
    compare("jet/h_jet_eta", h1(ak.to_numpy(ev["jet_eta"]), w, "jet_eta")[0])
    compare("jet/h_jet_m", h1(ak.to_numpy(ev["jet_m"]), w, "jet_m")[0])
    for lvl, _, _ in LEVELS:
        v, wv = pooled(lvl, "lnkt")
        compare(f"lund_{lvl}/h_{lvl}_all_lnkt", h1(v, wv, "lnkt")[0])
        n = ak.to_numpy(ak.num(ev[f"{lvl}_lnkt"]))
        compare(f"lund_{lvl}/h_{lvl}_nsplit", h1(n, w, "mult")[0])
        for i in range(NSPLIT_SHOW):
            v, wv = nth(lvl, "lnz", i)
            compare(f"lund_{lvl}/split{i:02d}/h_{lvl}_s{i:02d}_lnz", h1(v, wv, "lnz")[0])
    if HAS_AUX:
        compare("aux/h_x_mg", h1(ak.to_numpy(ev["x_mg"]), w, "x_mg")[0])
        if HAS_SEC:
            s = ak.to_numpy(ev["x_nsec"]) > 0
            compare("aux/h_x_kt_sec_max",
                    h1(ak.to_numpy(ev["x_kt_sec_max"])[s], w[s], "x_kt_sec_max")[0])

    for path, d in rows:
        print(f"  {'ok ' if d == 0 else 'DIFF'}  max|Δbin| = {d:<12g} {path}")
    print(f"\n{len(rows)} histograms compared, worst bin difference {worst:g}")